# 01 Stock Data

Start here. Choose the research settings. Later modules load the saved settings and intermediate files from data/processed/current/. The default files are the repository’s existing inputs; their historical universe bias remains explicitly recorded.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Imports and saved run


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'research_config.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the Pairs_trading repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.project_io import OUTPUT_DIR, initialize, load_config, load_frame, save_frame, save_json


## 2. Settings

All model settings are chosen here. Each execution overwrites the current settings and input copies. After changing settings or inputs, rerun this notebook and the following modules in order.


In [ ]:
from src.research_config import ResearchConfig
CONFIG = ResearchConfig()
PRICE_FILE = ROOT / 'data/processed/prices.parquet'
RATE_FILE = ROOT / 'data/processed/risk_free_rates.parquet'
BENCHMARK_FILE = ROOT / 'data/processed/systematic_risk/sp500_prices.parquet'
MEMBERSHIP_FILE = None  # Optional dated membership CSV paired with a full historical price panel.
ALLOW_LEGACY_UNIVERSE = True  # Acknowledge the included universe's known selection bias.
initialize(CONFIG, PRICE_FILE, RATE_FILE, BENCHMARK_FILE,
                                 membership=MEMBERSHIP_FILE, allow_legacy=ALLOW_LEGACY_UNIVERSE)
cfg = CONFIG
display(pd.Series(cfg.to_dict(), name='Research settings'))


## 3. Load and split prices

The first 70% of sessions form the training sample. Missingness is evaluated only there; test-period prices are never used to select assets. No live download occurs.


In [ ]:
from src.research_data import prepare_prices
prices = pd.read_parquet(OUTPUT_DIR / 'inputs/prices.parquet')
membership = pd.read_csv(OUTPUT_DIR / 'inputs/membership.csv') if (OUTPUT_DIR / 'inputs/membership.csv').exists() else None
train_prices, test_prices, availability = prepare_prices(prices, cfg, membership, ALLOW_LEGACY_UNIVERSE)
display(pd.DataFrame({'observations': [len(train_prices), len(test_prices)],
                      'start': [train_prices.index.min(), test_prices.index.min()],
                      'end': [train_prices.index.max(), test_prices.index.max()]}, index=['Formation', 'Out of sample']))
display(availability.head(10))


## 4. Inspect and save

These new files go into this run’s folder. Old thesis output files are preserved.


In [ ]:
save_frame('train_prices', train_prices)
save_frame('test_prices', test_prices)
save_frame('availability', availability)
ax = (train_prices.iloc[:, :5] / train_prices.iloc[0, :5]).plot(figsize=(10, 4), title='Formation prices normalized to one')
ax.set_ylabel('Normalized price')
plt.show()


## Save module completion

Wait for this confirmation before moving to the next notebook.


In [ ]:
print(f'Completed. Files saved in {OUTPUT_DIR}')
